# GEMC Quickstart in Binder
<hr style="height:4px;border:0;background:#4a90e2;">

This notebook reproduces the GEMC [Quickstart](https://gemc.github.io/home/documentation/quickstart/) documentation.

Run the cells in order with `Shift + Enter`. The notebook creates a `counter` system with a custom methane-gas target and a `G4_AIR` flux detector, builds its geometry, runs 10,000 events with CSV and JSON output, plots the digitized `totEdep` distribution from CSV, and shows the JSON event output.


In [ ]:
import subprocess, shutil, sys, warnings
from pathlib import Path
from IPython.display import Image, display

import vtk
import pyvista as pv
vtk.vtkObject.GlobalWarningDisplayOff()

from run_geometry import run_geometry

sys.path.insert(0, str(Path.cwd().parent))
from notebook_tools import edit


## Create the `counter` system

This is the same template command used in the documentation.

In [ ]:
counter_dir = Path("counter")
if counter_dir.exists():
    shutil.rmtree(counter_dir)

result = subprocess.run(["system_template.py", "-s", "counter"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)

materials_path = counter_dir / "materials.py"
materials_path.write_text('''from gmaterial import GMaterial

def define_materials(configuration):

	# example of material: methane gas, defined with number of atoms
	gmaterial = GMaterial("methaneGas")
	gmaterial.description = "methane gas CH4 0.000667 g/cm3"
	gmaterial.density = 0.000667
	gmaterial.addNAtoms("C", 1)
	gmaterial.addNAtoms("H", 4)
	gmaterial.publish(configuration)



''')

geometry_path = counter_dir / "geometry.py"
geometry_path.write_text('''from gvolume import GVolume

# These are example of methods to organize volumes creation

def build_counter(configuration):
	build_flux_box(configuration)
	build_target(configuration)

def build_flux_box(configuration):
	gvolume = GVolume("flux_box")
	gvolume.description = "air flux box"
	gvolume.make_box(40.0, 40.0, 2.0)
	gvolume.set_position(0, 0, 100)
	gvolume.material    = "G4_AIR"
	gvolume.color       = "3399FF"
	gvolume.style       = 1
	gvolume.digitization = "flux"
	gvolume.set_identifier("box", 2)  # identifier for this box
	gvolume.publish(configuration)

def build_target(configuration):
	gvolume = GVolume("target")
	gvolume.description = "methane gas target"
	gvolume.make_tube(0, 20, 40, 0, 360)
	gvolume.material    = "methaneGas"
	gvolume.publish(configuration)



''')

yaml_path = counter_dir / "counter.yaml"
text = yaml_path.read_text()
text = text.replace("nthreads: 4", "nthreads: 1")
text = text.replace("format: ascii", "format: csv")
if "format: json" not in text:
    text = text.replace(
        "gstreamer:\n  - filename: counter\n    format: csv",
        "gstreamer:\n  - filename: counter\n    format: csv\n  - filename: counter\n    format: json",
    )
yaml_path.write_text(text)

%cd counter


## Optional: inspect or edit the generated files

In [ ]:
edit("counter.yaml")


In [ ]:
edit("geometry.py")


In [ ]:
edit("materials.py")


## Build and display the geometry

Running `counter.py` creates `gemc.db` with the generated geometry and materials. The geometry is displayed with PyVista.

In [ ]:
run_geometry("counter.py")


## Run 10,000 events in GEMC

The generated `counter.yaml` writes CSV and JSON output. `-n=10000` gives enough statistics for the `totEdep` plot, and `-nthreads=1` keeps the output in `counter_t0_digitized.csv` and `counter_t0.json`.

To also write ROOT output, add another `gstreamer` entry with `format: root` in `counter.yaml`.


In [ ]:
driver = "-g4view=[{driver: TOOLSSG_OFFSCREEN}]"
camera = "-g4camera=[{phi: 0*deg, theta: 270*deg}]"

result = subprocess.run(
    ["gemc", "counter.yaml", driver, camera, "-n=10000", "-nthreads=1"],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print("GEMC failed:")
    print(result.stderr)
    raise SystemExit(result.returncode)

print(result.stdout)

image_file = Path("gemc_run_0.png")
if image_file.exists():
    display(Image(filename=str(image_file)))

for path in sorted(Path.cwd().glob("counter_t0*")):
    print(path)


## Show JSON output

The JSON streamer writes one event-structured file per thread. This cell shows the beginning of `counter_t0.json`.

In [ ]:
json_file = Path("counter_t0.json")
if json_file.exists():
    lines = json_file.read_text().splitlines()
    print("\n".join(lines[:12]))
else:
    print(f"No JSON output found: {json_file}")


## Plot total energy deposited

The flux digitization writes `totEdep`, the total energy deposited for each digitized hit.

In [ ]:
from analyzer import read_output, plot_variable

df = read_output("counter_t0_digitized.csv", kind="csv")
plot_variable(df, "totEdep", data="digitized", bins=50, show=True)
